## Imports

In [39]:
# Importar librerías necesarias para web scraping, manejo de emails, bases de datos y parseo de HTML
import json
import os
import cloudscraper
import yagmail

from sqlalchemy import create_engine as sa_create_engine, MetaData, Table, Column, Integer, String, Float, DateTime
from datetime import datetime
from dotenv import load_dotenv
from bs4 import BeautifulSoup


## Funciones

### Create Engine

In [53]:
# Crear una conexión a la base de datos y definir la tabla de productos
# Lee las credenciales del archivo .env, crea el engine de SQLAlchemy con encoding UTF-8
# Define la estructura de la tabla 'Object' con campos: id, fecha de creación, nombre, enlace, precio y tipo de producto
def create_db_engine():

    load_dotenv()

    url = os.getenv("DATABASE_URL")

    engine = sa_create_engine(url, connect_args={"options": "-c client_encoding=utf8"})
    metadata = MetaData()


    products = Table("Object", metadata, 
        Column('id', Integer, primary_key=True),
        Column('created_at', DateTime, default=datetime.utcnow),
        Column('product_name', String),
        Column('product_link', String),
        Column('product_price', String),
        Column('product_type', String),   
        Column('product_img', String)    
    )

    return engine, products


### Write JSON

In [41]:
# Guardar los datos extraídos en un archivo JSON
# Recibe un diccionario de objetos y un nombre de elemento
# Crea un archivo con formato 'datos_{element}.json' con indentación y caracteres especiales preservados
def write_json(dic_objets, element):

    with open(f'datos_{element}.json', 'w', encoding='utf-8') as archivo:
        json.dump(dic_objets, archivo, indent=4, ensure_ascii=False)

### Delete Data

In [42]:
# Eliminar todos los datos de la tabla de la base de datos
# Abre una conexión a la BD y ejecuta un DELETE para limpiar todos los registros
def delete_data(engine, table):

    with engine.connect() as conn:
        conn.execute(table.delete())
        conn.commit()

### Parser Data

In [43]:
# Extraer información de productos del HTML parseado
# Busca el contenedor principal y luego todos los elementos de productos dentro
# Para cada producto extrae: nombre, enlace, precio y tipo
# Retorna una lista de diccionarios con los datos del producto
def parser_data(htmlBody, type, params):

    clases = params["clases_objetos"]

    div_principal = clases["div_class_general"]
    div_objeto = clases["div_objetos"]
    div_imagenes = clases["div_imagenes"]
    div_src = clases["div_src"]

    diccionarioObjetos = []

    divObjetos = htmlBody.find("div", class_=div_principal)

    productos = divObjetos.select("a", class_=div_objeto)

    for element in productos:

        div_img = element.find("div", class_=div_imagenes) 
        img_source = div_img.find("div", class_=div_src)
        img_tag = img_source.find("img")

        img = img_tag.get("src")
        titulo = element.get("data-product-name")
        enlace = element.get("href")
        precio = element.get("data-product-price")
    
        productoJson = {
            "name" : titulo,
            "link" : enlace,
            "price" : precio,
            "type" : type,
            "img" : img
        }


        diccionarioObjetos.append(productoJson)

    return diccionarioObjetos

### Get HTML Body

In [44]:
# Descargar el contenido HTML de una URL usando cloudscraper para eludir protecciones
# Realiza una solicitud GET a la URL
# Si es exitosa (código 200), parsea el HTML y retorna un objeto BeautifulSoup
def get_html_body(url: string):

    scraper = cloudscraper.create_scraper()
    response = scraper.get(url)

    if response.status_code == 200:

        htmlBody = response.text

        soup = BeautifulSoup(htmlBody, "html.parser")

    return soup

### Clean Data

In [45]:
# Limpiar y normalizar datos de texto para evitar problemas de encoding
# Si el texto es None retorna vacío, sino lo convierte a UTF-8 reemplazando caracteres inválidos
def clean_data(text):
    if text is None: return ""
    return str(text).encode('utf-8','replace').decode('utf-8')

### Insert Data

In [51]:
# Insertar un registro de producto en la base de datos
# Limpia los datos de nombre y enlace, luego ejecuta un INSERT con todos los campos del producto
# Abre conexión, inserta datos y hace commit a la BD
def insert_data(name, link, price, type, img, engine, table):
    with engine.connect() as conn:

        name = clean_data(name)
        link = clean_data(link)

        query = table.insert().values(
            product_name = name,
            product_link = link,
            product_price = price,
            product_type = type,
            product_img = img
        )

        conn.execute(query)
        conn.commit()

### Get Info

In [47]:
# Cargar datos de un archivo JSON desde la carpeta 'data'
# Lee el archivo con el nombre especificado y retorna su contenido como diccionario
def get_info(nombreJson):

    with open(f'../data/{nombreJson}.json', 'r', encoding='utf-8') as archivo:
        data = json.load(archivo)

    return data

### Send Email

In [48]:
# Enviar un email notificando el resultado de la ejecución del scraping
# Carga credenciales de Gmail desde .env, obtiene la fecha actual
# Convierte el número del mes al nombre en español usando datos del JSON
# Se conecta a Gmail y envía un email con el resultado (Exitosa/Fallida)
def send_email(execution_type):

    # Credenciales
    load_dotenv()
    gmail = os.getenv("GMAIL")
    password = os.getenv("PASSWORD")
    date = datetime.now()

    # Convertir mes
    meses = get_info("month")
    mes_numero = str(date.month)
    mes_letra = meses[mes_numero]

    # Conectarme a mi correo
    yag = yagmail.SMTP(gmail,password)

    yag.send(
        to = "mariotirado2003@gmail.com",
        subject= "Ejecución diaria",
        contents= f"Ejecución día {date.day} de {mes_letra} {execution_type}"
    )

## Core

In [55]:
# ===== PROCESO PRINCIPAL (CORE) =====
# 1. Crear la conexión a la base de datos y obtener la tabla
engine,table = create_db_engine()

# 2. Limpiar datos antiguos: borrar todos los registros de la tabla
delete_data(engine,table)

# 3. Cargar configuración desde JSON: parámetros de scraping y URLs
params_exe = get_info("pcComponentesData")
paginas = params_exe["paginas"]
execution = False

# 4. Iterar por cada categoría de productos (tarjetas, ram, disco duro, monitores)
for element in paginas:

    # Obtener URL de la categoría
    url = paginas[element]["url"]
    # Descargar HTML de la página
    body_html = get_html_body(url)
    # Extraer datos de productos del HTML
    dic_objets = parser_data(body_html, element, params_exe)

    print(f"Cargando objetos {dic_objets[0]['type']}")

    # 5. Para cada producto encontrado, insertarlo en la base de datos
    for element_objects in dic_objets:

        element_name = element_objects['name']
        element_link = element_objects['link']
        element_price = element_objects['price']
        element_type = element_objects['type']
        element_img = element_objects['img']

        try:
            insert_data(element_name,element_link,element_price,element_type,element_img,engine,table)
            execution = True
        except Exception as e:
            execution = False
            print(f"error al subir los datos {e}")

# 6. Enviar email notificando el resultado de la ejecución
if execution : 
    send_email("Exitosa")
else:
    send_email("Fallida")


Cargando objetos tarjetas
Cargando objetos ram
Cargando objetos disco_duro
Cargando objetos monitores
Cargando objetos teclados
Cargando objetos sillas
Cargando objetos ratones
Cargando objetos auriculares
